# Task B: Inheritance

You have been given a ready‑made binary classification pipeline. The attached baseline reports `ROC‑AUC ≈ 0.97` on its validation set.

Each row corresponds to an event involving a certain entity at the time specified by event_time. In production, the model is applied to later events. Among the future data, there are both known entities and groups, as well as those that have not been observed before.

The numerical features f_00–f_23 were obtained using the existing feature pipeline. Its documentation is incomplete, so the correctness of individual features and the original validation protocol cannot be guaranteed.

Your goal is to achieve the highest possible score on the hidden future period. You are allowed to change:
* the validation, 
* the set of features, 
* preprocessing, 
* the model, 
* the training method.

## files:
* `train.csv             —` features and target;
* `test.csv              —` a later period without target;
* `starter.py            —` the original baseline;
* `sample_submission.csv —` a sample submission;
* `requirements.txt      —` minimal environment.

## main columns:
* `row_id —` row identifier;
* `event_time —` event time in arbitrary seconds;
* `entity_id, group_id —` recurring categorical identifiers;
* `cat_00–cat_04 —` categorical features;
* `f_00–f_23 —` anonymous numerical features;
* `target —` binary target variable, available only in `train`.

**NB!:** Rows in `train.csv` are shuffled. Rows in `test.csv` belong to a period subsequent to `train`.

## submission format
You must submit only a CSV file:

row_id,prediction

10000001,0.031

10000002,0.772

## requirements:

1) exactly one row for each `row_id` from `test.csv`;
2) columns must be named `row_id` and `prediction`;
3) prediction must be a finite number between `0` and `1`;
4) extra, missing, or duplicate `row_id` values are not allowed;
5) no text answers or code need to be submitted.

## metric
The hidden test is divided into three consecutive time blocks. Let `A1`, `A2`, `A3` denote the `ROC‑AUC` on these blocks.

A cold‑start subset is defined separately: rows where `entity_id` or `group_id` did not appear in train. Let `Acold` denote the `ROC‑AUC` on this subset.

### robust quality $R$ is calculated first:

$$
R \;=\; 0.50 \cdot mean(A_1, A_2, A_3) \;+\; 0.25 \cdot A_{cold} \;+\; 0.25 \cdot min(A_1, A_2, A_3, A_{cold})
$$

### final score:

$$
Score \;=\; 100 \;\cdot\; clip\left(\,\frac{(R - 0.55)}{0.20}, \, 0, \, 1\right)
$$

### benchmarks:
* `0 points  :` R ≤ 0.55;
* `50 points :` R = 0.65;
* `75 points :` R = 0.70;
* `100 points:` R ≥ 0.75.

The higher the score, the better. The best submission counts.

# Deliberately naive starter baseline

In [12]:
import pandas as pd
from catboost import CatBoostClassifier
from sklearn.metrics import roc_auc_score
from sklearn.model_selection import train_test_split

In [13]:
train = pd.read_csv("../data/data_B/train.csv")
train.info()

<class 'pandas.DataFrame'>
RangeIndex: 140035 entries, 0 to 140034
Data columns (total 34 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   row_id      140035 non-null  int64  
 1   event_time  140035 non-null  int64  
 2   entity_id   140035 non-null  int64  
 3   group_id    140035 non-null  int64  
 4   cat_00      140035 non-null  int64  
 5   cat_01      140035 non-null  int64  
 6   cat_02      140035 non-null  int64  
 7   cat_03      140035 non-null  int64  
 8   cat_04      140035 non-null  int64  
 9   f_00        140035 non-null  float64
 10  f_01        140035 non-null  float64
 11  f_02        140035 non-null  float64
 12  f_03        140035 non-null  float64
 13  f_04        140035 non-null  float64
 14  f_05        140035 non-null  float64
 15  f_06        140035 non-null  float64
 16  f_07        140035 non-null  float64
 17  f_08        140035 non-null  float64
 18  f_09        140035 non-null  float64
 19  f_10        1

In [14]:
train.target.value_counts(dropna=False, normalize=True)

target
0    0.974449
1    0.025551
Name: proportion, dtype: float64

In [15]:
train.row_id.value_counts(dropna=False)

row_id
10082466    1
10022591    1
10133054    1
10092682    1
10070012    1
           ..
10033930    1
10045529    1
10120962    1
10080876    1
10118087    1
Name: count, Length: 140035, dtype: int64

In [16]:
test = pd.read_csv("../data/data_B/test.csv")
test.info()

<class 'pandas.DataFrame'>
RangeIndex: 63270 entries, 0 to 63269
Data columns (total 33 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   row_id      63270 non-null  int64  
 1   event_time  63270 non-null  int64  
 2   entity_id   63270 non-null  int64  
 3   group_id    63270 non-null  int64  
 4   cat_00      63270 non-null  int64  
 5   cat_01      63270 non-null  int64  
 6   cat_02      63270 non-null  int64  
 7   cat_03      63270 non-null  int64  
 8   cat_04      63270 non-null  int64  
 9   f_00        63270 non-null  float64
 10  f_01        63270 non-null  float64
 11  f_02        63270 non-null  float64
 12  f_03        63270 non-null  float64
 13  f_04        63270 non-null  float64
 14  f_05        63270 non-null  float64
 15  f_06        63270 non-null  float64
 16  f_07        63270 non-null  float64
 17  f_08        63270 non-null  float64
 18  f_09        63270 non-null  float64
 19  f_10        63270 non-null  float64


In [17]:
test.row_id.value_counts(dropna=False)

row_id
10140035    1
10140036    1
10140037    1
10140038    1
10140039    1
           ..
10203300    1
10203301    1
10203302    1
10203303    1
10203304    1
Name: count, Length: 63270, dtype: int64

In [18]:
features = [c for c in test.columns if c != "row_id"]
#features

In [19]:
cat_features = ["entity_id", "group_id"] + [c for c in features if c.startswith("cat_")]
cat_features

['entity_id', 'group_id', 'cat_00', 'cat_01', 'cat_02', 'cat_03', 'cat_04']

In [20]:
for frame in (train, test):
    for col in cat_features:
        frame[col] = frame[col].astype(str)

train.info()

<class 'pandas.DataFrame'>
RangeIndex: 140035 entries, 0 to 140034
Data columns (total 34 columns):
 #   Column      Non-Null Count   Dtype  
---  ------      --------------   -----  
 0   row_id      140035 non-null  int64  
 1   event_time  140035 non-null  int64  
 2   entity_id   140035 non-null  str    
 3   group_id    140035 non-null  str    
 4   cat_00      140035 non-null  str    
 5   cat_01      140035 non-null  str    
 6   cat_02      140035 non-null  str    
 7   cat_03      140035 non-null  str    
 8   cat_04      140035 non-null  str    
 9   f_00        140035 non-null  float64
 10  f_01        140035 non-null  float64
 11  f_02        140035 non-null  float64
 12  f_03        140035 non-null  float64
 13  f_04        140035 non-null  float64
 14  f_05        140035 non-null  float64
 15  f_06        140035 non-null  float64
 16  f_07        140035 non-null  float64
 17  f_08        140035 non-null  float64
 18  f_09        140035 non-null  float64
 19  f_10        1

In [21]:
train_idx, valid_idx = train_test_split(
    range(len(train)),
    test_size=0.25,
    random_state=42,
    stratify=train["target"],
)

In [22]:
model = CatBoostClassifier(
    iterations=140,
    depth=7,
    learning_rate=0.11,
    loss_function="Logloss",
    eval_metric="AUC",
    l2_leaf_reg=6,
    random_seed=42,
    verbose=False,
    allow_writing_files=False,
)

In [23]:
model.fit(
    train.loc[train_idx, features],
    train.loc[train_idx, "target"],
    cat_features=[features.index(c) for c in cat_features],
)

CatBoostClassifier(allow_writing_files=False, depth=7, eval_metric='AUC', iterations=140, l2_leaf_reg=6, learning_rate=0.11, loss_function='Logloss', random_seed=42, verbose=False)

In [24]:
valid_prediction = model.predict_proba(train.loc[valid_idx, features])[:, 1]

In [25]:

print(f"Random-split ROC-AUC: {roc_auc_score(train.loc[valid_idx, 'target'], valid_prediction):.6f}")

Random-split ROC-AUC: 0.970494


In [26]:

model.fit(
    train[features],
    train["target"],
    cat_features=[features.index(c) for c in cat_features],
)

CatBoostClassifier(allow_writing_files=False, depth=7, eval_metric='AUC', iterations=140, l2_leaf_reg=6, learning_rate=0.11, loss_function='Logloss', random_seed=42, verbose=False)

In [27]:
test_prediction = model.predict_proba(test[features])[:, 1]

pd.DataFrame({"row_id": test["row_id"], "prediction": test_prediction}).to_csv(
                                       "../results/submission.csv", index=False)

print('FYI: submission.csv has been saved OK!')


FYI: submission.csv has been saved OK!
